# Individual Exercise: Mini Data Quality Audit

Homework notebook for Week 2.


## Instructions

Perform a mini data quality audit and submit this completed notebook.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
data_path ="https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/refs/heads/main/data/raw/week2_customer_transactions_messy.csv"
df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df.head(15)


Loaded: https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/refs/heads/main/data/raw/week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## Required tasks

1. describe the dataset briefly
2. identify issues by dimension
3. compute at least 3 KPIs
4. define at least 3 validation rules
5. create a short audit summary
6. recommend cleaning steps for next chapter (do not implement full pipelines)


## Task 1 - Dataset description

### Your answer
##### customer_id contains a missing value.
##### transaction_date is inconsistent, with multiple formats used instead of a standardized one.
##### amount includes zero and negative values, which may represent refunds, failed transactions, or data errors.
##### currency indicates multi-currency transactions but also contains inconsistent labeling (e.g., "EUR" vs "EURO").
##### payment_method contains categorical values but shows inconsistency in formatting (e.g., "card" vs "CARD").
##### status includes multiple values (e.g., completed, pending), representing different stages of the transaction lifecycle.
##### region represents country codes, but values are inconsistently formatted (e.g., uppercase vs lowercase).
##### last_updated generally follows a consistent format, but there is no actual missing value in the sample.

## Task 2 - Issues by dimension


In [3]:
issue_table = pd.DataFrame([['Missing customer_id','Completeness','Impacts customer analytics'],['Duplicate transaction_id','Uniqueness','May double count revenue']], columns=['Issue','Dimension','Impact'])
issue_table


,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Duplicate transaction_id,Uniqueness,May double count revenue


## Task 3 - KPI calculations


In [4]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))
kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()
amount=pd.to_numeric(df['amount'], errors='coerce')
date_ok=pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate']=(amount.isna() | (amount<0) | ~date_ok).mean()
pd.DataFrame({'KPI':list(kpis.keys()), 'Value':list(kpis.values())})


,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727


### Your KPI interpretation

####Completeness Rate (0.9596 ≈ 95.96%)
About 96% of the data fields are filled and not missing. Only a small portion of values are missing.

####Duplication Rate (0.0909 ≈ 9.09%)
Around 9% of the records are duplicates. This indicates some repeated transactions or records that may need deduplication.

####Error Rate (0.2727 ≈ 27.27%)
About 27% of the data contains errors such as invalid formats, inconsistent values, or rule violations. This is relatively high and suggests significant data quality issues.


## Task 4 - Validation rules


In [5]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce')<0,
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T


,affected_rows
transaction_id_required,0
amount_non_negative,1
transaction_date_valid,1


## Task 5 - Audit summary


In [14]:
audit_summary = pd.DataFrame([['Example issue',0,'Medium','Example next action']], columns=['issue_type','affected_rows','severity','recommended_next_action'])
audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Example issue,0,Medium,Example next action


In [15]:
audit_summary = pd.DataFrame([
    ['Missing customer_id', df['customer_id'].isna().sum(), 'High', 'Impute or remove records'],
    ['Duplicate transaction_id', df.duplicated(subset=['transaction_id']).sum(), 'High', 'Remove duplicates'],
    ['Invalid amount values', pd.to_numeric(df['amount'], errors='coerce').isna().sum(), 'High', 'Clean and standardize amount'],
    ['Negative amount values', (pd.to_numeric(df['amount'], errors='coerce') < 0).sum(), 'Medium', 'Validate business rules'],
    ['Invalid transaction_date', pd.to_datetime(df['transaction_date'], errors='coerce').isna().sum(), 'Medium', 'Standardize date format']
], columns=['issue_type','affected_rows','severity','recommended_next_action'])

audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Missing customer_id,1,High,Impute or remove records
1,Duplicate transaction_id,1,High,Remove duplicates
2,Invalid amount values,1,High,Clean and standardize amount
3,Negative amount values,1,Medium,Validate business rules
4,Invalid transaction_date,3,Medium,Standardize date format


## Task 6 - Recommended cleaning steps for next chapter

- Recommendation 1
- Recommendation 2
- Recommendation 3
- Recommendation 4


## Reflection questions

1. Which KPI gave the strongest signal?
2. Which issue should be escalated first?
3. What additional metadata would improve this audit?


## Bonus (recommended): write your own audit function

Create at least one helper function with clear parameters and docstring.
This demonstrates that your logic is reusable and understandable.


In [ ]:
def summarize_rule_violations(rule_dictionary):
    """Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks.

    Returns
    -------
    pd.DataFrame
        Table with one row per rule and violation counts.
    """
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    }).sort_values('affected_rows', ascending=False)

# Example usage with your `rules` dictionary:
summarize_rule_violations(rules)


### Explain your function parameters

In 3-5 lines, explain:
- what each parameter means,
- why you selected default values,
- how changing parameters affects KPI/audit results.


## Submission checklist

- [ ] Dataset described
- [ ] Issues mapped to dimensions
- [ ] At least 3 KPIs calculated
- [ ] At least 3 validation rules defined
- [ ] Audit summary completed
- [ ] Cleaning recommendations proposed (not implemented)
